# 04 — Slice-aware DLinear on ETTh1

This notebook tests the next DLinear improvements after ordinary RevIN: **future-mean prediction**, a **learned per-channel gate**, and—only if validation supports it—**future-scale prediction**. DLinear's moving-average decomposition and temporal linear projections remain the forecasting backbone.

Protocol: ETTh1 multivariate forecasting, 336 input hours, 96 forecast hours, seed 2021, and the same train/validation/test partitions and optimizer settings as the reconstruction notebook.

## 1. Hypothesis and variants

The 336-hour input is divided into fourteen 24-hour slices. A small channel-wise linear head maps their observed means to four future daily means. The normalized DLinear output models the within-window pattern, while the predicted slice means reconstruct an evolving baseline:

$$\hat Y = \sigma_X DLinear((X-\mu_X)/\sigma_X) + \hat\mu_{future}. $$

The gated model also computes raw DLinear and blends the two forecasts independently for each channel:

$$\hat Y_c=(1-g_c)\hat Y_{raw,c}+g_c\hat Y_{slice,c},\qquad g_c=\operatorname{sigmoid}(a_c).$$

The training loss is forecast MSE plus 0.1 times future-slice-statistics MSE. True future statistics are labels during training only; validation and test inference uses the input window alone.

In [ ]:
from pathlib import Path
import copy, math

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

from ts_project.data import FEATURE_COLUMNS, build_window_datasets, prepare_etth1
from ts_project.models import (
    DLinear, GatedSliceAwareDLinear, RevINDLinear, SliceAwareDLinear,
    slice_statistics_loss,
)
from ts_project.training import seed_everything

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SEED, INPUT_LENGTH, PREDICTION_LENGTH, CHANNELS = 2021, 336, 96, 7
BATCH_SIZE, SLICE_LENGTH = 32, 24
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = prepare_etth1(PROJECT_ROOT / 'data' / 'raw' / 'ETTh1.csv')
datasets = build_window_datasets(data, input_length=INPUT_LENGTH, prediction_length=PREDICTION_LENGTH)
print('Device:', device)

In [ ]:
def make_backbone():
    return DLinear(
        input_length=INPUT_LENGTH, prediction_length=PREDICTION_LENGTH,
        channels=CHANNELS, moving_average=25, individual=False,
    )

def make_loaders():
    generator = torch.Generator().manual_seed(SEED)
    return {
        'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, generator=generator),
        'validation': DataLoader(datasets['validation'], batch_size=BATCH_SIZE),
        'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE),
    }

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    squared = torch.zeros(CHANNELS, dtype=torch.float64)
    absolute = torch.zeros(CHANNELS, dtype=torch.float64)
    count = 0
    for x, y in loader:
        prediction = model(x.to(device)).cpu()
        error = prediction - y
        squared += (error ** 2).sum(dim=(0, 1)).double()
        absolute += error.abs().sum(dim=(0, 1)).double()
        count += y.shape[0] * y.shape[1]
    channel_mse, channel_mae = squared / count, absolute / count
    return {
        'MSE': channel_mse.mean().item(),
        'MAE': channel_mae.mean().item(),
        'RMSE': math.sqrt(channel_mse.mean().item()),
        'per_channel_MSE': dict(zip(FEATURE_COLUMNS, channel_mse.tolist())),
    }

In [ ]:
def train_variant(name, model, auxiliary_weight=0.1):
    seed_everything(SEED)
    loaders = make_loaders()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    best_state, best_validation, best_epoch, bad_epochs = None, float('inf'), 0, 0
    history = []

    for epoch in range(1, 11):
        model.train()
        squared_sum = element_count = 0
        for x, y in loaders['train']:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            if hasattr(model, 'forward_with_statistics'):
                prediction, statistics = model.forward_with_statistics(x)
                loss = torch.mean((prediction - y) ** 2) + auxiliary_weight * slice_statistics_loss(
                    statistics, y, slice_length=SLICE_LENGTH
                )
            else:
                prediction = model(x)
                loss = torch.mean((prediction - y) ** 2)
            loss.backward()
            optimizer.step()
            squared_sum += ((prediction.detach() - y) ** 2).sum().item()
            element_count += y.numel()

        validation_mse = evaluate(model, loaders['validation'])['MSE']
        history.append({'epoch': epoch, 'train_mse': squared_sum / element_count, 'validation_mse': validation_mse})
        print(f'{name:28s} epoch {epoch:2d}: validation MSE={validation_mse:.6f}')
        if validation_mse < best_validation:
            best_state, best_validation, best_epoch, bad_epochs = copy.deepcopy(model.state_dict()), validation_mse, epoch, 0
        else:
            bad_epochs += 1
            if bad_epochs >= 3:
                break
        for group in optimizer.param_groups:
            group['lr'] = 0.005 * (0.5 ** (epoch - 1))

    model.load_state_dict(best_state)
    result = {
        'best_epoch': best_epoch, 'best_validation_MSE': best_validation,
        'parameters': sum(p.numel() for p in model.parameters()),
        'history': history, 'test': evaluate(model, loaders['test']),
    }
    if isinstance(model, GatedSliceAwareDLinear):
        result['gates'] = dict(zip(FEATURE_COLUMNS, model.gates.detach().cpu().tolist()))
    return result

## 2. Run paired variants

Every model is initialized and trained from the same seed with a newly seeded shuffled loader. Step 6 is validation-gated: it runs only if step 4 or 5 beats paired DLinear on validation MSE.

In [ ]:
variants = {
    'DLinear': make_backbone(),
    'RevIN-DLinear': RevINDLinear(make_backbone()),
    'Slice-aware mean': SliceAwareDLinear(make_backbone(), slice_length=SLICE_LENGTH),
    'Gated slice-aware mean': GatedSliceAwareDLinear(
        make_backbone(), SliceAwareDLinear(make_backbone(), slice_length=SLICE_LENGTH)
    ),
}
results = {name: train_variant(name, model) for name, model in variants.items()}

best_mean_validation = min(
    results['Slice-aware mean']['best_validation_MSE'],
    results['Gated slice-aware mean']['best_validation_MSE'],
)
if best_mean_validation < results['DLinear']['best_validation_MSE']:
    scale_model = GatedSliceAwareDLinear(
        make_backbone(),
        SliceAwareDLinear(make_backbone(), slice_length=SLICE_LENGTH, predict_scale=True),
    )
    results['Gated slice-aware mean+scale'] = train_variant('Gated slice-aware mean+scale', scale_model)
    step_6_decision = 'ran'
else:
    step_6_decision = 'skipped: neither mean-aware variant improved validation MSE'
step_6_decision

## 3. Measured result

Executed on 2026-08-06 with the repository's ETTh1 data and CPU PyTorch 2.12.1.

| Model | Validation MSE | Test MSE | Test MAE | Parameters | Change vs DLinear |
|---|---:|---:|---:|---:|---:|
| DLinear | 0.648873 | 0.371544 | 0.393440 | 64,704 | — |
| RevIN-DLinear | 0.678503 | 0.384905 | 0.403304 | 64,718 | 3.60% worse |
| Slice-aware mean | 0.700887 | 0.372975 | 0.394345 | 65,124 | 0.39% worse |
| Gated slice-aware mean | 0.678392 | 0.375075 | 0.396327 | 129,835 | 0.95% worse |

The mean-aware models did not beat DLinear on validation, so future-scale prediction (step 6) was not run. This prevents test performance from deciding model development.

In [ ]:
summary = pd.DataFrame({
    name: {
        'Validation MSE': result['best_validation_MSE'],
        'Test MSE': result['test']['MSE'],
        'Test MAE': result['test']['MAE'],
        'Parameters': result['parameters'],
    }
    for name, result in results.items()
}).T
summary['MSE change vs DLinear (%)'] = 100 * (summary['Test MSE'] / summary.loc['DLinear', 'Test MSE'] - 1)
summary

## 4. Channel diagnosis

The aggregate negative result hides a strong temperature benefit. The table below reports every feature separately. **Positive percentages mean lower MSE than DLinear; negative percentages mean deterioration.**

| Feature | DLinear MSE | RevIN MSE | RevIN improvement | Slice-aware MSE | Slice-aware improvement | Gated MSE | Gated improvement |
|---|---:|---:|---:|---:|---:|---:|---:|
| HUFL | 0.749956 | 0.791241 | -5.51% | 0.755203 | -0.70% | 0.751802 | -0.25% |
| HULL | 0.208271 | 0.214804 | -3.14% | 0.207452 | +0.39% | 0.207498 | +0.37% |
| MUFL | 0.774865 | 0.820111 | -5.84% | 0.782200 | -0.95% | 0.779100 | -0.55% |
| MULL | 0.169163 | 0.174676 | -3.26% | 0.168842 | +0.19% | 0.169546 | -0.23% |
| LUFL | 0.507095 | 0.518808 | -2.31% | 0.515072 | -1.57% | 0.526915 | -3.91% |
| LULL | 0.120640 | 0.121216 | -0.48% | 0.126610 | -4.95% | 0.125047 | -3.65% |
| **OT** | **0.070816** | **0.053478** | **+24.48%** | **0.055442** | **+21.71%** | **0.065618** | **+7.34%** |

`OT` is the only feature improved by all three normalization variants. Slice-aware mean also gives very small gains on `HULL` (+0.39%) and `MULL` (+0.19%), but these are too small to treat as established improvements from one seed. The four deteriorating load features—especially `LUFL` and `LULL`—outweigh the `OT` gain in the overall average.

The learned gate did not solve this automatically. Its final values ranged from 0.41 to 0.74 and remained above 0.5 for most load variables, even though validation favored raw DLinear overall. The gated model also doubled the backbone parameter count. This indicates that an unconstrained end-to-end sigmoid gate is not reliable channel selection under the present objective.

In [ ]:
channel_mse = pd.DataFrame({
    name: result['test']['per_channel_MSE'] for name, result in results.items()
})
model_names = ['RevIN-DLinear', 'Slice-aware mean', 'Gated slice-aware mean']
improvement = pd.DataFrame({
    name: 100 * (1 - channel_mse[name] / channel_mse['DLinear'])
    for name in model_names
})
display(channel_mse.round(6))
display(improvement.round(2).style.format('{:+.2f}%').background_gradient(cmap='RdYlGn', vmin=-25, vmax=25))
channel_mse[['DLinear', 'RevIN-DLinear', 'Slice-aware mean', 'Gated slice-aware mean']].plot.bar(
    figsize=(11, 4), ylabel='Test MSE', title='Per-channel ETTh1 error'
)
plt.tight_layout()
plt.show()
ax = improvement.plot.bar(
    figsize=(11, 4), ylabel='MSE improvement vs DLinear (%)',
    title='Per-feature improvement over DLinear', color=['#4C78A8', '#F58518', '#54A24B']
)
ax.axhline(0, color='black', linewidth=0.8)
ax.bar_label(ax.containers[0], fmt='%.1f%%', fontsize=8)
ax.bar_label(ax.containers[1], fmt='%.1f%%', fontsize=8)
ax.bar_label(ax.containers[2], fmt='%.1f%%', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Decision

Steps 4 and 5 are informative negative results, not improvements to claim. Step 6 should remain deferred. The next justified experiment is **validation-selected channel masking**: train raw and normalized/slice-aware candidates, select the method for each channel from validation only, freeze that mask, and evaluate test once. This directly tests the strong `OT`/load split without asking a poorly identified sigmoid gate to discover it during joint optimization.

If that selection generalizes across multiple seeds and horizons, it can then motivate a constrained or regularized learned selector.